In [17]:
# Cell 1 — Imports
import requests
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

import os
os.environ["WDM_SSL_VERIFY"] = "0"

import sys


import re
import time
import pandas as pd

from selenium import webdriver
from selenium.webdriver.edge.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.microsoft import EdgeChromiumDriverManager
from selenium.webdriver.common.action_chains import ActionChains


def log(message: str) -> None:
    print(message, flush=True)


In [18]:
# Cell 2 — Configuration
URL = "https://platform.atheneum-app.com/clientV2/p/c7ac4c6fe39c8b91d4917288f0026e7f4ffe1d32e460edd39df12b341271f60c/available"

print(URL)


https://platform.atheneum-app.com/clientV2/p/c7ac4c6fe39c8b91d4917288f0026e7f4ffe1d32e460edd39df12b341271f60c/available


In [19]:
# Cell 3 — Browser setup with download folder
DOWNLOAD_DIR = os.getcwd()

def create_driver():
    log("Preparing Edge driver...")

    options = webdriver.EdgeOptions()
    options.add_argument("--ignore-certificate-errors")
    options.add_argument("--start-maximized")

    prefs = {
        "download.default_directory": DOWNLOAD_DIR,
        "download.prompt_for_download": False,
        "download.directory_upgrade": True,
        "safebrowsing.enabled": True,
    }
    options.add_experimental_option("prefs", prefs)

    driver_path = EdgeChromiumDriverManager().install()
    log(f"Using Edge driver: {driver_path}")

    driver = webdriver.Edge(
        service=Service(driver_path),
        options=options
    )

    driver.set_page_load_timeout(45)
    driver.set_script_timeout(45)

    return driver

In [20]:
# 4a - close popup

def close_login_popup_if_present(driver):
    close_labels = [
        "Close",
        "×",
        "X",
        "Cancel",
        "Not now",
        "Continue without login"
    ]

    for label in close_labels:
        try:
            xpath = (
                "//*[self::button or self::a or @role='button']"
                f"[contains(translate(., "
                "'ABCDEFGHIJKLMNOPQRSTUVWXYZ', "
                "'abcdefghijklmnopqrstuvwxyz'), "
                f"'{label.lower()}')]"
            )

            btn = WebDriverWait(driver, 3).until(
                EC.element_to_be_clickable((By.XPATH, xpath))
            )

            driver.execute_script("arguments[0].click();", btn)
            log(f"Closed popup using: {label}")
            time.sleep(2)
            return True

        except Exception:
            continue

    # Fallback: press Escape
    try:
        ActionChains(driver).send_keys("\ue00c").perform()
        log("Pressed Escape to close popup")
        time.sleep(1)
        return True
    except Exception:
        return False
    

    driver = None

    try:
        driver = create_driver()

        log("Opening Atheneum page...")
        driver.get(url)

        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.TAG_NAME, "body"))
        )

        log("Page loaded. Waiting for controls...")
        time.sleep(5)

        close_login_popup_if_present(driver)

        driver.execute_script("window.scrollTo(0, 0);")
        time.sleep(1)

        # Save diagnostic dump
        body_text = driver.find_element(By.TAG_NAME, "body").text

        with open("atheneum_page_dump.txt", "w", encoding="utf-8") as f:
            f.write(body_text)

        log("Saved diagnostic dump: atheneum_page_dump.txt")

        clicked = click_atheneum_download(driver)

        if not clicked:
            with open("atheneum_html_dump.html", "w", encoding="utf-8") as f:
                f.write(driver.page_source)

            raise Exception(
                "Could not find Excel download control. "
                "Saved atheneum_html_dump.html for inspection."
            )

        downloaded_file = wait_for_download(DOWNLOAD_DIR)

        output_file = os.path.join(DOWNLOAD_DIR, "atheneum_experts.xlsx")

        if downloaded_file != output_file:
            os.replace(downloaded_file, output_file)

        log(f"Saved output: {output_file}")

        return output_file

    except Exception as e:
        log(f"Fatal error: {e}")
        return None

    finally:
        if driver is not None:
            driver.quit()

In [21]:
# cell 4b - click logic

# Cell 5 — Download button detection
def click_visible_download_option(driver):
    labels = [
        "Excel",
        "XLSX",
        "Export",
        "Download",
        "Download Excel",
        "Export Excel",
        "Export to Excel",
        "Download experts",
        "Export experts",
        "Candidate export",
        "Expert export"
    ]

    for label in labels:
        try:
            xpath = (
                "//*[contains(translate(normalize-space(.), "
                "'ABCDEFGHIJKLMNOPQRSTUVWXYZ', "
                "'abcdefghijklmnopqrstuvwxyz'), "
                f"'{label.lower()}')]"
            )

            elems = driver.find_elements(By.XPATH, xpath)

            for elem in elems:
                if elem.is_displayed():
                    driver.execute_script(
                        "arguments[0].scrollIntoView({block: 'center'});",
                        elem
                    )

                    time.sleep(0.5)

                    driver.execute_script(
                        "arguments[0].click();",
                        elem
                    )

                    log(f"Clicked download option: {label}")

                    time.sleep(2)

                    return True

        except Exception:
            continue

    return False


def click_atheneum_download(driver):
    menu_xpaths = [
        "//*[contains(@class, 'download')]",
        "//*[contains(@class, 'export')]",
        "//*[contains(@class, 'excel')]",
        "//*[contains(@class, 'xlsx')]",
        "//*[contains(@aria-label, 'download')]",
        "//*[contains(@aria-label, 'export')]",
        "//*[contains(@title, 'download')]",
        "//*[contains(@title, 'export')]",
        "//button[contains(., '...')]",
        "//button[contains(., '⋮')]",
        "//button[contains(., 'More')]",
        "//*[@role='button'][contains(., '⋮')]",
    ]

    for xpath in menu_xpaths:
        try:
            elems = driver.find_elements(By.XPATH, xpath)

            for elem in elems:
                if elem.is_displayed():
                    driver.execute_script(
                        "arguments[0].scrollIntoView({block: 'center'});",
                        elem
                    )

                    time.sleep(0.5)

                    driver.execute_script(
                        "arguments[0].click();",
                        elem
                    )

                    log(f"Clicked possible menu/download control: {xpath}")

                    time.sleep(2)

                    if click_visible_download_option(driver):
                        return True

        except Exception:
            continue

    return click_visible_download_option(driver)


In [22]:
# Cell 4c — Download Atheneum Excel export
def wait_for_download(download_dir: str, timeout: int = 60):
    log("Waiting for Excel download...")

    start = time.time()

    while time.time() - start < timeout:
        files = os.listdir(download_dir)

        partials = [
            f for f in files
            if f.endswith(".crdownload") or f.endswith(".tmp")
        ]

        excel_files = [
            f for f in files
            if f.lower().endswith((".xlsx", ".xls"))
        ]

        if excel_files and not partials:
            latest_file = max(
                [os.path.join(download_dir, f) for f in excel_files],
                key=os.path.getctime
            )
            log(f"Downloaded file found: {latest_file}")
            return latest_file

        time.sleep(1)

    raise TimeoutError("No completed Excel download found.")


def download_atheneum_experts_excel(url: str):
    driver = None

    try:
        driver = create_driver()

        log("Opening Atheneum page...")
        driver.get(url)

        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.TAG_NAME, "body"))
        )

        log("Page loaded. Waiting for controls...")
        time.sleep(5)

        close_login_popup_if_present(driver)

        # Save diagnostic page text after popup is closed
        body_text = driver.find_element(By.TAG_NAME, "body").text
        with open("atheneum_page_dump.txt", "w", encoding="utf-8") as f:
            f.write(body_text)

        log("Saved diagnostic dump: atheneum_page_dump.txt")

        # Try common export/download button labels
        download_labels = [
            "Export",
            "Download",
            "Excel",
            "XLSX",
            "CSV",
            "Export experts",
            "Download experts",
            "Export to Excel",
            "Download Excel"
        ]

        clicked = False

        for label in download_labels:
            try:
                xpath = (
                    "//*[self::button or self::a or @role='button']"
                    f"[contains(translate(., "
                    "'ABCDEFGHIJKLMNOPQRSTUVWXYZ', "
                    "'abcdefghijklmnopqrstuvwxyz'), "
                    f"'{label.lower()}')]"
                )

                btn = WebDriverWait(driver, 5).until(
                    EC.element_to_be_clickable((By.XPATH, xpath))
                )

                driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", btn)
                time.sleep(0.5)
                driver.execute_script("arguments[0].click();", btn)

                log(f"Clicked download/export control: {label}")
                clicked = True
                break

            except Exception:
                continue

        if not clicked:
            raise Exception(
                "Could not find an Export / Download / Excel button. "
                "Check atheneum_page_dump.txt and inspect the page controls."
            )

        downloaded_file = wait_for_download(DOWNLOAD_DIR)

        # Standardise filename
        output_file = os.path.join(DOWNLOAD_DIR, "atheneum_experts.xlsx")

        if downloaded_file != output_file:
            os.replace(downloaded_file, output_file)

        log(f"Saved output: {output_file}")

        return output_file

    except Exception as e:
        log(f"Fatal error: {e}")
        return None

    finally:
        if driver is not None:
            driver.quit()


In [ ]:
# Cell 5 — Run download
def main():
    log("Starting Atheneum Excel download...")

    file_path = download_atheneum_experts_excel(URL)

    if file_path:
        df = pd.read_excel(file_path)
        log(f"Rows in downloaded Excel: {len(df)}")
        return df

    return pd.DataFrame()


df = main()
df

Starting scrape_ATH.py...


NameError: name 'fetch_all_experts' is not defined